# 📥 Download Full Ethereum Transaction Data

This notebook downloads the **complete** Ethereum transaction dataset from BigQuery (no sampling).

**What you'll get:**
- ~1-1.5 million transactions per day
- ~8-10 million transactions for 7 days
- Saved as Parquet files to Google Drive

**Instructions:**
1. Run each cell in order
2. Authenticate when prompted
3. Wait for export to complete (~15-30 min)
4. Download files from Google Drive

In [ ]:
# Step 1: Install dependencies
!pip install -q google-cloud-bigquery db-dtypes pandas pyarrow
print("✅ Dependencies installed")

In [ ]:
# Step 2: Authenticate with Google
from google.colab import auth
auth.authenticate_user()
print("✅ Authenticated")

In [ ]:
# Step 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = "/content/drive/MyDrive/eth_full_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Files will be saved to: {OUTPUT_DIR}")

In [ ]:
# Step 4: Configuration
#
# ⚙️ EDIT THESE VALUES:
#

DAYS_TO_FETCH = 7          # Number of days to fetch (1-30)
MIN_VALUE_ETH = 0.01       # Minimum transaction value (filters dust/spam)
                           # Set to 0 to include ALL transactions

print(f"📅 Will fetch: {DAYS_TO_FETCH} days")
print(f"💰 Minimum value: {MIN_VALUE_ETH} ETH")

In [ ]:
# Step 5: Estimate data size
from google.cloud import bigquery
from datetime import datetime, timedelta

client = bigquery.Client()

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=DAYS_TO_FETCH)

value_filter = ""
if MIN_VALUE_ETH > 0:
    value_filter = f"AND CAST(value AS FLOAT64) / 1e18 >= {MIN_VALUE_ETH}"

estimate_query = f"""
SELECT 
    COUNT(*) as total_tx,
    ROUND(SUM(CAST(value AS FLOAT64)) / 1e18, 2) as total_eth,
    COUNT(DISTINCT from_address) as senders,
    COUNT(DISTINCT to_address) as receivers
FROM `bigquery-public-data.crypto_ethereum.transactions`
WHERE 
    block_timestamp >= TIMESTAMP('{start_date.strftime('%Y-%m-%d')}')
    AND block_timestamp < TIMESTAMP('{end_date.strftime('%Y-%m-%d')}')
    AND to_address IS NOT NULL
    {value_filter}
"""

print("📊 Counting transactions...")
result = list(client.query(estimate_query).result())[0]

print(f"\n{'='*50}")
print(f"📈 DATA ESTIMATE")
print(f"{'='*50}")
print(f"Date range: {start_date.date()} to {end_date.date()}")
print(f"Total transactions: {result.total_tx:,}")
print(f"Total ETH moved: {result.total_eth:,.2f}")
print(f"Unique senders: {result.senders:,}")
print(f"Unique receivers: {result.receivers:,}")
print(f"Est. file size: ~{result.total_tx * 400 / (1024**3):.2f} GB")
print(f"{'='*50}")

In [ ]:
# Step 6: Export data (day by day)
import pandas as pd
import gc

def export_day(date):
    """Export one day of transactions."""
    date_str = date.strftime('%Y-%m-%d')
    next_date_str = (date + timedelta(days=1)).strftime('%Y-%m-%d')
    output_file = f"{OUTPUT_DIR}/eth_{date_str}.parquet"
    
    # Skip if exists
    if os.path.exists(output_file):
        df = pd.read_parquet(output_file)
        print(f"  {date_str}: Already downloaded ({len(df):,} rows)")
        return len(df)
    
    query = f"""
    SELECT
        `hash` as tx_hash,
        block_number,
        block_timestamp,
        from_address,
        to_address,
        CAST(value AS FLOAT64) / 1e18 as value_eth,
        CAST(gas_price AS FLOAT64) / 1e9 as gas_price_gwei,
        receipt_gas_used as gas_used,
        gas as gas_limit,
        nonce,
        transaction_index
    FROM `bigquery-public-data.crypto_ethereum.transactions`
    WHERE 
        block_timestamp >= TIMESTAMP('{date_str}')
        AND block_timestamp < TIMESTAMP('{next_date_str}')
        AND to_address IS NOT NULL
        {value_filter}
    ORDER BY block_number, transaction_index
    """
    
    print(f"  {date_str}: Downloading...")
    df = client.query(query).to_dataframe()
    
    if len(df) == 0:
        print(f"  {date_str}: No data")
        return 0
    
    df.to_parquet(output_file, compression='snappy', index=False)
    size_mb = os.path.getsize(output_file) / (1024**2)
    print(f"  {date_str}: ✅ {len(df):,} transactions ({size_mb:.1f} MB)")
    
    del df
    gc.collect()
    return len(df)

# Run export
print("\n🚀 STARTING DOWNLOAD\n")
total = 0
current = start_date

while current < end_date:
    total += export_day(current)
    current += timedelta(days=1)

print(f"\n{'='*50}")
print(f"✅ DOWNLOAD COMPLETE")
print(f"{'='*50}")
print(f"Total transactions: {total:,}")
print(f"Location: {OUTPUT_DIR}")

In [ ]:
# Step 7: List downloaded files
print("\n📁 Downloaded files:\n")
total_size = 0
total_rows = 0

for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith('.parquet'):
        fpath = os.path.join(OUTPUT_DIR, f)
        size_mb = os.path.getsize(fpath) / (1024**2)
        total_size += size_mb
        
        df = pd.read_parquet(fpath)
        total_rows += len(df)
        print(f"  {f}: {len(df):,} rows, {size_mb:.1f} MB")
        del df

print(f"\n{'='*50}")
print(f"TOTAL: {total_rows:,} transactions, {total_size:.1f} MB")
print(f"{'='*50}")

In [ ]:
# Step 8: Preview sample data
files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.parquet')]
if files:
    sample = pd.read_parquet(os.path.join(OUTPUT_DIR, files[0]))
    print("📋 Sample data:\n")
    print(f"Columns: {list(sample.columns)}\n")
    display(sample.head(10))

---
## 📥 Next Steps

1. **Go to Google Drive** → `My Drive` → `eth_full_data`
2. **Download** all `.parquet` files
3. **Copy** to your local machine: `c:\amttp\data\eth_full\`
4. **Run** the batch processing pipeline:

```powershell
python scripts/batch_fraud_pipeline.py --input c:\amttp\data\eth_full --output c:\amttp\processed
```